In [ ]:
%pip install scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
from scipy.interpolate import griddata

# 1. Import 18 data points from excel
data = np.array([
      # Circle Points (negative electrode)
      [15, 10, 2.146], [15, 11, 2.929], [14, 11, 2.674], [14, 10, 2.636], [13, 10, 1.751], [13, 9, 3.191], [14, 9, 2.655], [15, 9, 3.450], 

      # Bar Points (positive electrode)
      [23, 3, 11.01], [21, 3, 11.54], [19, 3, 11.45], [17, 3, 10.52], [15, 3, 9.65], [13, 3, 9.14], [11, 3, 8.81], [9, 3, 9.07], [7, 3, 9.77], [5, 3, 9.75]
    
])

x_measured = data[:, 0]
y_measured = data[:, 1]
v_measured = data[:, 2]

# 2. Create Lab Grid Space
x_grid = np.linspace(0, 26, 200)
y_grid = np.linspace(0, 20, 200)
X, Y = np.meshgrid(x_grid, y_grid)

#3. Linear Interpolation
V = griddata((x_measured, y_measured), v_measured, (X, Y), method ='linear')

# Fill outer edges with boundary voltage
V[Y <= 3] = np.nanmean(v_measured[y_measured == 3])
V[(X >= 13) & (X <= 15) & (Y >= 9) & (Y <= 11) & np.isnan(V)] = np.nanmean(v_measured[y_measured != 3])
V = griddata((x_measured, y_measured), v_measured, (X, Y), method='nearest')

# Create 3D Surface pyplotfig = plt.figure(figsize=(10, 8), dpi=300)
fig = plt.figure(figsize=(10, 8), dpi=300)
ax = fig.add_subplot(111, projection='3d')


# Plot surface with high contrast
surf = ax.plot_surface(X, Y, V, cmap='plasma', edgecolor='none', alpha=0.9)

# Add Color Bar Scale to Side
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)
cbar.set_label('Electric Potential (VDC)', fontsize=10, fontweight='bold')

# Axis styling to match canvas grid
ax.set_xlim(0, 26)
ax.set_ylim(0, 20)
ax.set_zlim(0, 13)
ax.set_xlabel('X-Coordinate (cm)', fontsize=10, fontweight='bold', labelpad=10)
ax.set_ylabel('Y-Coordinate (cm)', fontsize=10, fontweight='bold', labelpad=10)
ax.set_zlabel('Voltage (V)', fontsize=10, fontweight='bold', labelpad=10)
ax.set_title('3-D Surface Plot of Electric Potential (Part C)', fontsize=12, fontweight='bold', pad=20)

# Rotate the view angle so 3D peak is perfectly visible
ax.view_init(elev=25, azim=-45)

# Save high-res PNG to folder
plt.savefig('3d_potential_surface.png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Grid Set up Matching Canvas Grid
x_min, x_max = 0, 26
y_min, y_max = 0, 20

# Use a coarser grid so the arrows don't overlap
x_range = np.arange(0, 27, 1.0)
y_range = np.arange(0, 21, 1.0)
X, Y = np.meshgrid(x_range, y_range)

# 2. Electrode coordinates from Part C
x_center, y_center = 14.0, 10.0
y_bar = 3.5

# 3. Calculate electric field vectors
# Flip signs to match experiment
r1_sq = (X - x_center)**2 + (Y - y_center)**2 + 1e-5
r2_sq = (X - x_center)**2 + (Y - y_center)**2 + (Y - (2 * y_bar - y_center))**2 + 1e-5

# Field points AWAY from positive bar and TOWARD negative circle
Ex = -4.5 * ((X - x_center)/r1_sq -(X - x_center)/r2_sq)
Ey = -4.5 * ((Y - y_center)/r1_sq - (Y - (2*y_bar - y_center))/r2_sq)

# 4. Calculate magnitude in V/m
E_magnitude = np.sqrt(Ex**2 + Ey**2) * 100

# Mask out vectors inside physical electrodes
inside_circle = (X - x_center)**2 + (Y - y_center)**2 < 1.0
inside_bar = (Y <= 1.1) & (X >= 5) & (X <= 23)
Ex[inside_circle | inside_bar] = np.nan
Ey[inside_circle | inside_bar] = np.nan

# 5. Color array
# Red: >50 V/m, Orange: 5 to 50 V/m, Gray: <5 V/m
colors = np.empty(X.shape, dtype=object)
colors[E_magnitude > 50] = '#e74c3c'
colors[(E_magnitude >= 5) & (E_magnitude <= 50)] = '#f39c12'
colors[E_magnitude < 5] = '#95a5a6'

# Flatten arrays for the quiver plotting tool
xf, yf = X.flatten(), Y.flatten()
exf, eyf = Ex.flatten(), Ey.flatten()
colors_f = colors.flatten()

# Filter out the NaN values from the isnide of electrodes
valid = ~np.isnan(exf)
xf, yf, exf, eyf, colors_f = xf[valid], yf[valid], exf[valid], eyf[valid], colors_f[valid]

# 6. Plotting the Vector Map
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
ax.set_facecolor('#ffffff')

# Draw the electrodes
circle = plt.Circle((x_center, y_center), 0.8, color='#2c3e50', zorder=5, label='Negative Circle (-)')
ax.add_patch(circle)
ax.plot([5, 23], [1.0, 1.0], color='#2c3ef0', linewidth=10, solid_capstyle='butt', zorder=5, label='Positive Bar (+)')

# Draw vector arrows
ax.quiver(xf, yf, exf, eyf, color=colors_f, scale=120, width=0.004, headwidth=4, headlength=5, zorder=3)

# Formatting to match canvas template
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xticks(np.arange(0, 27, 2))
ax.set_yticks(np.arange(0, 21, 2))
ax.set_xlabel('X-Coordinate (cm)', fontsize=11, fontweight='bold')
ax.set_ylabel('Y-Coordinate (cm)', fontsize=11, fontweight='bold')
ax.set_title('Electric Field Vector Map (Magnitude & Direction)\nColor Coded by Strength (V/m)', fontsize=12, fontweight='bold', pad=15)
ax.grid(True, which='both', color='#e0e0e0', linestyle='--', linewidth=0.5)
ax.set_aspect('equal')

# Create a clean manual custom legend
ax.plot([0], [0], marker='>', color='none', markerfacecolor='#e74c3c', markeredgecolor='#e74c3c', markersize=10, label='Strong Field (>50 V/m)')
ax.plot([0], [0], marker='>', color='none', markerfacecolor='#f39c12', markeredgecolor='#f39c12', markersize=10, label='Moderate Field (5-50 V/m)')
ax.plot([0], [0], marker='>', color='none', markerfacecolor='#95a5a6', markeredgecolor='#95a5a6', markersize=10, label='Weak Field (<5 V/m)')

ax.legend(loc='upper right', framealpha=0.9)


# Save the final vector plot image
plt.savefig('electric_field_vectors.png', bbox_inches='tight', dpi=300)
plt.show()

    

